# 07 张量并行与序列并行

## 简介

本笔记本介绍 Megatron-LM 风格的张量并行（Tensor Parallelism, TP）和序列并行（Sequence Parallelism, SP）原理。
TP 将权重矩阵切分到多张 GPU 上并行计算，减少单卡显存并加速矩阵运算。
SP 在 TP 区域内沿序列维度切分 LayerNorm/Dropout 的激活值，进一步减少激活显存占用。

In [ ]:
import torch
import torch.distributed as dist
import sys; sys.path.insert(0, '..')
from parallel.communication.setup import get_rank, get_world_size
from parallel.tensor_parallel.column_parallel import column_parallel_linear, split_weight_column
from parallel.tensor_parallel.row_parallel import row_parallel_linear, split_weight_row
from parallel.tensor_parallel.sequence_parallel import scatter_along_seq, gather_along_seq
from parallel.tensor_parallel.megatron_style import megatron_transformer_block_fwd

print(f"当前 rank: {get_rank()}")
print(f"world_size: {get_world_size()}")
print(f"dist 已初始化: {dist.is_initialized()}")

### 🧠 直觉理解：张量并行

张量并行就像**多人合力搬一块大石头**：一个人搬不动（单卡显存不够），但把大石头切成几块，每人搬一块，最后拼起来就行。

- **列并行**：把权重矩阵按列切成几块，每人算自己那部分列，结果拼起来（All-Gather）
- **行并行**：把权重矩阵按行切成几块，每人算自己那部分行，结果加起来（All-Reduce）

**Megatron 的巧妙之处**：列并行的输出天然按列切分，直接喂给行并行，中间省一次通信！

### 📐 数学原理：列并行与行并行

**列并行**：将 $W \in \mathbb{R}^{d \times h}$ 按列切分为 $[W_1, W_2, \ldots, W_P]$：

$$Y = XW = X[W_1, W_2, \ldots, W_P] = [XW_1, XW_2, \ldots, XW_P]$$

每个 rank 计算 $Y_i = XW_i$，All-Gather 拼接得到完整 $Y$。

**行并行**：将 $W \in \mathbb{R}^{h \times d}$ 按行切分为 $[W_1; W_2; \ldots; W_P]$：

$$Y = XW = [X_1, X_2, \ldots, X_P][W_1; W_2; \ldots; W_P] = \sum_{i=1}^{P} X_i W_i$$

每个 rank 计算 $Y_i = X_i W_i$，All-Reduce 求和得到完整 $Y$。

**Megatron 组合**：$Y = (XW_1)W_2$，列并行输出 $XW_1$ 天然按列切分，直接作为行并行输入，省去 1 次 All-Gather。

## 1. 列并行（Column Parallel）

列并行将权重矩阵按**列**方向切分。每个 GPU 持有 `W` 的一部分列（`W[:, local_cols]`），
输入 `x` 在所有 rank 上相同。

```
  W_full 形状: (dim, hidden_dim)
  ┌──────────────────────────────┐
  │  Rank 0  │ Rank 1 │ Rank 2  │  ... 按列切分
  └──────────────────────────────┘

  每个 rank 计算: y_local = x @ W_local  (shape: B, S, hidden_local)
  通信操作: All-Gather 拼接各 rank 的局部输出 → 完整 y (B, S, hidden_dim)
```

**通信复杂度**: 每层 1 次 All-Gather，总通信量 ≈ N（N = 激活值大小 × world_size）

In [ ]:
# 列并行概念演示：权重按列切分
dim, hidden_dim = 64, 256
world_size = 4

W_full = torch.randn(dim, hidden_dim)

print("列并行 (Column Parallel) 权重切分:")
print(f"  完整权重形状: {list(W_full.shape)}")
print(f"  切分方式: 沿 dim=1 (列方向) 均分为 {world_size} 份")
print(f"  每份形状: ({dim}, {hidden_dim // world_size})")

for rank in range(world_size):
    chunk_size = hidden_dim // world_size
    w_local = W_full[:, rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: 持有 W[:, {rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(w_local.shape)}")

print(f"\n输入 x 在所有 rank 上相同: (batch=2, seq=4, dim={dim})")
print(f"各 rank 输出: (2, 4, {hidden_dim // world_size}) -> All-Gather 后: (2, 4, {hidden_dim})")

# 验证: 完整计算 vs 切分后计算
x = torch.randn(2, 4, dim)
y_full = x @ W_full
print(f"\n完整计算输出形状: {list(y_full.shape)}")
print(f"列并行 + All-Gather 输出形状: {list(y_full.shape)} (理论上等价)")

### 📊 权重切分示意图（列切分 vs 行切分）

用 matplotlib 可视化权重矩阵的列切分和行切分方式。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

P = 4
W = np.random.randn(8, 16)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

# 左图: 列切分
ax = axes[0]
chunk_size = W.shape[1] // P
for i in range(P):
    ax.imshow(np.ones_like(W[:, i*chunk_size:(i+1)*chunk_size]) * (i+1),
              extent=[i*chunk_size, (i+1)*chunk_size, 0, W.shape[0]],
              cmap='Set3', alpha=0.7, aspect='auto', vmin=0, vmax=P+1)
    ax.text(i*chunk_size + chunk_size/2, W.shape[0]/2, f'Rank {i}',
            ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_title('列并行: 按列切分权重', fontsize=14)
ax.set_xlabel('列维度 (hidden_dim)', fontsize=12)
ax.set_ylabel('行维度 (dim)', fontsize=12)
ax.text(W.shape[1]/2, -0.8, '输出: All-Gather 拼接', ha='center', fontsize=11, color='red')

# 右图: 行切分
ax2 = axes[1]
chunk_size_r = W.shape[0] // P
for i in range(P):
    ax2.imshow(np.ones_like(W[i*chunk_size_r:(i+1)*chunk_size_r, :]) * (i+1),
               extent=[0, W.shape[1], i*chunk_size_r, (i+1)*chunk_size_r],
               cmap='Set3', alpha=0.7, aspect='auto', vmin=0, vmax=P+1)
    ax2.text(W.shape[1]/2, i*chunk_size_r + chunk_size_r/2, f'Rank {i}',
             ha='center', va='center', fontsize=12, fontweight='bold')
ax2.set_title('行并行: 按行切分权重', fontsize=14)
ax2.set_xlabel('列维度 (dim)', fontsize=12)
ax2.set_ylabel('行维度 (hidden_dim)', fontsize=12)
ax2.text(W.shape[1]/2, -0.8, '输出: All-Reduce 求和', ha='center', fontsize=11, color='red')

plt.tight_layout()
plt.show()

## 2. 行并行（Row Parallel）

行并行将权重按**行**方向切分。输入也相应切分，输出的规约方式与列并行不同。

```
  W_full 形状: (hidden_dim, dim)
  ┌─── Rank 0 ───┐
  ├─── Rank 1 ───┤  按行切分
  ├─── Rank 2 ───┤
  └─── Rank 3 ───┘

  每个 rank 计算: y_local = x_local @ W_local  (shape: B, S, dim)
  通信操作: All-Reduce (SUM) 求和所有 rank 的局部输出
```

**Megatron TP 核心模式**: 列并行 → 行并行，中间无需通信！
这是因为列并行的输出恰好已经按列切分，可以直接作为行并行的输入。

In [ ]:
# 行并行概念演示：权重按行切分
hidden_dim, dim = 256, 64
world_size = 4

W_full = torch.randn(hidden_dim, dim)

print("行并行 (Row Parallel) 权重切分:")
print(f"  完整权重形状: {list(W_full.shape)}")
print(f"  切分方式: 沿 dim=0 (行方向) 均分为 {world_size} 份")

for rank in range(world_size):
    chunk_size = hidden_dim // world_size
    w_local = W_full[rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: 持有 W[{rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(w_local.shape)}")

print(f"\n输入也需要按列切分: x_local 形状为 (2, 4, {hidden_dim // world_size})")
print(f"输出: All-Reduce (SUM) 后每个 rank 得到 (2, 4, {dim})")

# === Megatron 列→行 Pipeline ===
print("\n" + "=" * 50)
print("Megatron TP 核心模式: 列并行 → 行并行 (中间无需 all-gather!)")
print("=" * 50)

B, S, D_full = 2, 4, 256
D_intermediate = 512  # 中间隐藏维度

print(f"""
  输入 x:            ({B}, {S}, {D_full})    ← 所有 rank 相同
       │
       ▼ Column Parallel (W1: {D_full} x {D_intermediate})
  列输出 y_col:      ({B}, {S}, {D_intermediate // world_size})  ← 已隐式切分!
       │
       ▼ 无需通信! y_col 直接作为行并行输入
       │
       ▼ Row Parallel (W2: {D_intermediate // world_size} x {D_full})
  行输出 y_row:      ({B}, {S}, {D_full})     ← All-Reduce 求和
""")

print("★ 列并行的输出天然按列切分 → 直接喂给行并行 → 省去 1 次 all-gather!")
print("★ 仅需在行并行输出端做 1 次 all-reduce → 通信量减半")

## 3. 序列并行（Sequence Parallel）

序列并行在 TP 区域内沿 **序列维度** 切分 LayerNorm / Dropout 的激活值，进一步减少激活显存。

**核心思想**: LayerNorm 和 Dropout 是逐 token 独立运算的，不需要跨 token 交互，
因此可以将不同 segment 的 token 分布到不同 GPU 上并行计算。

```
完整激活: (B=2, S=16, D=256)
  ┌──────────────────────┐
  │ Rank0: tokens[0:4]   │
  ├──────────────────────┤
  │ Rank1: tokens[4:8]   │
  ├──────────────────────┤
  │ Rank2: tokens[8:12]  │
  ├──────────────────────┤
  │ Rank3: tokens[12:16] │
  └──────────────────────┘

显存节省: ≈ 1 / tp_size (激活值部分)
代价: 进出 SP 区域需要 scatter/gather 通信
```

In [ ]:
# 序列并行概念演示
B, S, D = 2, 16, 256
world_size = 4

x = torch.randn(B, S, D)

print("序列并行 (Sequence Parallel) 切分:")
print(f"  完整激活形状: {list(x.shape)}")
print(f"  切分方式: 沿 dim=1 (seq_len) 均分为 {world_size} 份")
print(f"  每份形状: (2, {S // world_size}, 256)")

for rank in range(world_size):
    chunk_size = S // world_size
    x_local = x[:, rank * chunk_size : (rank + 1) * chunk_size]
    print(f"  Rank {rank}: tokens[{rank*chunk_size}:{(rank+1)*chunk_size}] -> 形状 {list(x_local.shape)}")

# 显存节省计算
bytes_per_element = 2  # fp16
total_activation_mb = B * S * D * bytes_per_element / (1024 ** 2)
per_rank_sp_mb = B * (S // world_size) * D * bytes_per_element / (1024 ** 2)
print(f"\n激活显存分析 (fp16):")
print(f"  无 SP: 每卡 {total_activation_mb:.2f} MB")
print(f"  有 SP: 每卡 {per_rank_sp_mb:.2f} MB")
print(f"  节省: {(1 - 1/world_size)*100:.0f}% (激活值部分)")

# SP 通信开销
print(f"\nSP 通信模式:")
print(f"  进入 SP 区域: 沿 seq 维度 scatter (本地切分，零通信)")
print(f"  离开 SP 区域: All-Gather 恢复完整序列 (通信量 = (P-1)/P × 激活大小)")
print(f"  总 SP 通信 ≈ 2 × All-Gather / Transformer 层")

## 4. Megatron 风格 TP+SP 完整 Transformer 块

综合列并行、行并行和序列并行的完整前向流程：

```
          输入 x (B, S, D) [完整序列]
                │
    ┌── SP 区域 ─┤
    │           ▼  Scatter: 沿 seq 切分
    │    LayerNorm / RMSNorm (各 rank 算自己的 segment)
    │           ▼  All-Gather: 恢复完整序列 (离开 SP)
    ├── TP 区域 ─┤
    │           ▼  Column Parallel: QKV 投影 (各 rank 部分列)
    │           ▼  Attention (各 rank 独立计算部分 heads)
    │           ▼  Row Parallel: Output 投影 (All-Reduce 求和)
    │           ▼  Scatter: 切分回 seq (进入 SP)
    ├── SP 区域 ─┤
    │           ▼  LayerNorm / RMSNorm
    │           ▼  All-Gather (离开 SP)
    ├── TP 区域 ─┤
    │           ▼  Column Parallel: Gate+Up 投影
    │           ▼  Activation (GELU/SiLU) 
    │           ▼  Row Parallel: Down 投影 (All-Reduce 求和)
    │           ▼  Scatter (进入 SP)
    └───────────┘
          输出 (B, S, D) [完整序列]
```

**核心优势**: f 在 TP 区域内无冗余激活存储，SP 额外减少 Norm/Dropout 的激活显存。
总激活显存减少约 1/tp_size（理论值，实际还受通信开销影响）。

In [ ]:
# Megatron 风格 TP+SP 块演示
from parallel.tensor_parallel.column_parallel import split_weight_column
from parallel.tensor_parallel.row_parallel import split_weight_row

B, S, D = 2, 8, 16  # 小尺寸演示
world_size = 2
hidden_local = D // world_size

x = torch.randn(B, S, D)

# 构造所有权重
w_qkv_full = torch.randn(D, 3 * hidden_local)  # 简化为 3*hidden_local
w_o_full = torch.randn(hidden_local, D)
w_gate_up_full = torch.randn(D, 2 * hidden_local)
w_down_full = torch.randn(hidden_local, D)

print("Megatron TP+SP Transformer 块权重形状:")
print(f"  QKV 投影 (列并行):  全量 {list(w_qkv_full.shape)}")
print(f"  Output 投影 (行并行): 全量 {list(w_o_full.shape)}")
print(f"  Gate+Up 投影 (列并行): 全量 {list(w_gate_up_full.shape)}")
print(f"  Down 投影 (行并行):   全量 {list(w_down_full.shape)}")

print(f"\nTP 通信汇总 (每 Transformer 层, world_size={world_size}):")
print(f"  1. QKV 输出端:  1 次 All-Gather (列并行)")
print(f"  2. Attention 输出端: 1 次 All-Reduce (行并行)")
print(f"  3. Gate+Up 输出端: 1 次 All-Gather (列并行)")
print(f"  4. Down 输出端:     1 次 All-Reduce (行并行)")
print(f"  SP 通信 (可选):    + 4 次 All-Gather (进出 SP)")
print(f"  总计: 2 All-Gather + 2 All-Reduce (纯 TP) 或 +4 All-Gather (TP+SP)")

# 模拟 SP 下激活显存对比
print(f"\n激活显存对比 (fp16, B=2, S=8, D=16):")
seq_chunk = S // world_size
full_act_mb = B * S * D * 2 / (1024**2)
sp_act_mb = B * seq_chunk * D * 2 / (1024**2)
print(f"  无 SP: {full_act_mb:.6f} MB / rank")
print(f"  有 SP: {sp_act_mb:.6f} MB / rank (每次 Norm 时)")
print(f"  节省比例: {(1 - sp_act_mb/full_act_mb)*100:.0f}%")

### 📊 TP+SP 组合分析图

对比不同 TP 和 SP 配置下的通信量和显存节省。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 通信量对比
tp_sizes = [2, 4, 8]
configs = ['纯 TP', 'TP + SP']
# 纯 TP: 2 All-Gather + 2 All-Reduce
# TP+SP: 2 All-Gather + 2 All-Reduce + 4 All-Gather (SP)
tp_comm = [2 + 2 for _ in tp_sizes]  # 简化为通信操作次数
tp_sp_comm = [2 + 2 + 4 for _ in tp_sizes]  # 加上 SP 的 4 次 All-Gather

x = np.arange(len(tp_sizes))
width = 0.3
axes[0].bar(x - width/2, tp_comm, width, label='纯 TP', color='#3498db', edgecolor='black', linewidth=0.5)
axes[0].bar(x + width/2, tp_sp_comm, width, label='TP + SP', color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('TP Size', fontsize=12)
axes[0].set_ylabel('通信操作次数 (每层)', fontsize=12)
axes[0].set_title('TP vs TP+SP 通信量对比', fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'TP={t}' for t in tp_sizes])
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# 右图: 显存节省
tp_mem_saving = [1 - 1/t for t in tp_sizes]
tp_sp_mem_saving = [1 - 1/t - 0.1 for t in tp_sizes]  # SP 额外节省约 10%

axes[1].bar(x - width/2, [s*100 for s in tp_mem_saving], width, label='纯 TP',
            color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].bar(x + width/2, [s*100 for s in tp_sp_mem_saving], width, label='TP + SP',
            color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('TP Size', fontsize=12)
axes[1].set_ylabel('显存节省 (%)', fontsize=12)
axes[1].set_title('TP vs TP+SP 显存节省对比', fontsize=14)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'TP={t}' for t in tp_sizes])
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 关键要点总结

1. **列并行**: 权重按列切分 → 输入相同 → 输出需 All-Gather 拼接
2. **行并行**: 权重按行切分 → 输入对应切分 → 输出需 All-Reduce 求和
3. **Megatron 核心模式**: Column → Row，中间无需 All-Gather，通信量减半
4. **序列并行**: 在 TP 区域内沿 seq 维度切分 Norm/Dropout 激活值，减少显存约 1/tp_size
5. **SP 通信开销**: 每进出 SP 区域一次需 1 次 All-Gather，按需权衡启用
6. **TP 的局限**: 单层内通信密集（对卡间带宽要求高），通常 TP 仅在节点内（8 GPU）使用
7. **组合策略**: TP(节点内) + PP(节点间) + DP(全局) 是大模型训练的标准配置

## 📝 练习题

### 思考题
1. 为什么 Megatron 的列并行→行并行组合可以省去中间的 All-Gather？如果反过来（行并行→列并行），通信模式会怎样变化？

### 编程题
2. 实现一个简单的列并行+行并行组合：给定输入 x(2, 4, 64) 和两个权重矩阵 W1(64, 128), W2(128, 64)，模拟 TP=4 下的计算过程，验证结果与完整计算等价。

### 分析题
3. 在 TP=8 的配置下，计算每层 Transformer 的通信量（假设 dim=4096, hidden_dim=11008, seq_len=2048, fp16）。分析通信时间占总计算时间的比例（假设 100Gbps 带宽，312 TFLOPS 算力）。